# Future-Guided Learning (Mackey-Glass) — Cải tiến kiến trúc RNN → LSTM/GRU

**Notebook tự chứa (self-contained)** — không cần clone repo, không cần upload file: tự
sinh dữ liệu Mackey-Glass, tự định nghĩa toàn bộ model, tự chạy so sánh. Chạy được ngay trên
Kaggle (Settings → Internet: **On** để `pip install jitcdde`; GPU không bắt buộc — xem ghi
chú runtime ở cuối notebook).

## Nguồn gốc & phạm vi

- **Tái hiện từ**: Gunasekaran et al., *"A predictive approach to enhance time-series
  forecasting"*, Nature Communications 2025. Repo gốc:
  github.com/SkyeGunasekaran/FutureGuidedLearning (pipeline Mackey-Glass, file
  `mackey_glass/exp/base_exp.py` + `mackey_glass/utils/utils.py`).
- **Phần KHÔNG đổi so với tác giả**: toàn bộ cơ chế Future-Guided Learning — quy trình
  3 giai đoạn Teacher (1-step) → Baseline (H-step, không FGL) → Student (H-step, có FGL),
  công thức loss chưng cất `α·CE + (1-α)·T²·KL(Teacher‖Student)`, cách sinh dữ liệu
  Mackey-Glass, cách rời rạc hoá (binning) thành bài toán phân loại.
- **Phần CẢI TIẾN tự đề xuất** (không có trong code/paper gốc, đánh dấu rõ bên dưới): tác
  giả chỉ dùng `nn.RNN` (vanilla) cho cả Teacher/Baseline/Student. Notebook này tham số hoá
  kiến trúc để chọn được **RNN / LSTM / GRU**, rồi so sánh mức cải thiện MSE (Baseline→
  Student) giữa 3 kiến trúc trên cùng 6 seed, nhằm kiểm tra: kiến trúc mạnh hơn có giúp FGL
  cải thiện nhiều hơn không.

**Kết quả đã chạy trước** (máy local, CPU, xem `repro/docs/EXPERIMENT_LOG.md` mục "Lần chạy
4"): RNN 26,42% → LSTM 33,96% → GRU 33,07% (trung bình giảm MSE, 6 seed). Notebook này giúp
chạy lại độc lập trên Kaggle để đối chiếu.


## 0. Cài đặt dependency

Chỉ `jitcdde` là gói ngoài — dùng để giải phương trình vi phân có trễ sinh chuỗi Mackey-Glass. Nếu lỗi biên dịch `symengine` khi cài, thử chạy lại cell này (đôi khi do cache pip trên Kaggle) hoặc khởi động lại kernel sau khi cài.

In [ ]:
!pip install -q jitcdde


In [ ]:
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from jitcdde import jitcdde, y, t, jitcdde_lyap
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 1. Sinh dữ liệu Mackey-Glass

Y hệt `MackeyGlass` trong `utils/utils.py` của bản tái hiện — phương trình vi phân có trễ:

```
dy/dt = β·y(t-τ) / (1 + y(t-τ)^n) - γ·y(t)
```

giải bằng `jitcdde`. Tham số và `seed_id=42` dùng đúng như `utils/generate.py` của tác giả
(đây là **seed sinh dữ liệu**, cố định, khác với **seed huấn luyện** ở Mục 4 — đúng thiết kế
gốc: dữ liệu `data.pkl` sinh 1 lần, dùng lại cho mọi lần train ở các seed khác nhau).

In [ ]:
class MackeyGlass(Dataset):
    """ Dataset cho chuỗi Mackey-Glass (y hệt utils/utils.py bản tái hiện). """
    def __init__(self, tau, constant_past, nmg=10, beta=0.2, gamma=0.1,
                 dt=1.0, splits=(8000., 2000.), start_offset=0., seed_id=0):
        super().__init__()
        self.tau, self.constant_past = tau, constant_past
        self.nmg, self.beta, self.gamma, self.dt = nmg, beta, gamma, dt
        self.traintime, self.testtime = splits
        self.start_offset, self.seed_id = start_offset, seed_id
        self.maxtime = self.traintime + self.testtime + self.dt
        self.traintime_pts = round(self.traintime / self.dt)
        self.testtime_pts = round(self.testtime / self.dt)
        self.maxtime_pts = self.traintime_pts + self.testtime_pts + 1
        self.mackeyglass_specification = [
            self.beta * y(0, t - self.tau) / (1 + y(0, t - self.tau) ** self.nmg) - self.gamma * y(0)
        ]
        self.generate_data()
        self.split_data()

    def generate_data(self):
        np.random.seed(self.seed_id)
        self.DDE = jitcdde_lyap(self.mackeyglass_specification)
        self.DDE.constant_past([self.constant_past])
        self.DDE.step_on_discontinuities()
        self.mackeyglass_soln = torch.zeros((self.maxtime_pts, 1), dtype=torch.float64)
        lyaps = torch.zeros((self.maxtime_pts, 1), dtype=torch.float64)
        lyaps_weights = torch.zeros((self.maxtime_pts, 1), dtype=torch.float64)
        count = 0
        for cur_t in torch.arange(self.DDE.t + self.start_offset,
                                   self.DDE.t + self.start_offset + self.maxtime,
                                   self.dt, dtype=torch.float64):
            value, lyap, weight = self.DDE.integrate(cur_t.item())
            self.mackeyglass_soln[count, 0] = value[0]
            lyaps[count, 0] = lyap[0]
            lyaps_weights[count, 0] = weight
            count += 1
        self.total_var = torch.var(self.mackeyglass_soln[:, 0], True)
        self.lyap_exp = ((lyaps.T @ lyaps_weights) / lyaps_weights.sum()).item()

    def split_data(self):
        self.ind_train = torch.arange(0, self.traintime_pts)
        self.ind_test = torch.arange(self.traintime_pts, self.maxtime_pts - 1)

    def __len__(self):
        return len(self.mackeyglass_soln) - 1

    def __getitem__(self, idx):
        sample = torch.unsqueeze(self.mackeyglass_soln[idx, :], dim=0)
        target = self.mackeyglass_soln[idx + 1, :]
        return sample, target


In [ ]:
MG_PARAMS = dict(
    tau=17,
    constant_past=0.9,
    dt=1.0,
    splits=(10000., 0.),
    seed_id=42,  # seed sinh DỮ LIỆU — cố định, không phải seed huấn luyện ở Mục 4
)

print("Đang sinh chuỗi Mackey-Glass (có thể mất vài chục giây)...")
_mg_dataset = MackeyGlass(**MG_PARAMS)

_series = []
for idx in range(len(_mg_dataset)):
    _, target = _mg_dataset[idx]
    _series.append(target.squeeze().item())

_col1 = torch.tensor(_series, dtype=torch.float64).unsqueeze(1)
_col2 = _col1.clone()
data = torch.cat((_col1, _col2), dim=1)
print("Data shape:", data.shape)


## 2. Kiến trúc mô hình — RNN / LSTM / GRU

> **⭐ Đây là phần cải tiến tự đề xuất.** Tác giả gốc chỉ có `nn.RNN` (vanilla) cứng trong
> code. Lớp dưới đây tham số hoá bằng `cell_type ∈ {rnn, lstm, gru}` — khi `cell_type='rnn'`
> hành vi giống hệt bản gốc (đã xác nhận cho ra đúng số liệu cũ khi chạy lại, xem
> `repro/ours/patches/arch_lstm_gru.patch`). Dùng **cùng kiến trúc cho cả ba vai trò**
> Teacher/Baseline/Student, đúng nguyên tắc paper — chỉ đổi loại cell.

In [ ]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=2, cell_type='rnn'):
        super(RNN, self).__init__()
        cell_type = cell_type.lower()
        self.cell_type = cell_type
        if cell_type == 'rnn':
            self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        elif cell_type == 'lstm':
            self.rnn = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        elif cell_type == 'gru':
            self.rnn = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        else:
            raise ValueError(f"Unknown cell_type: {cell_type!r} (expected 'rnn', 'lstm', or 'gru')")
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.rnn.num_layers, x.size(0), self.rnn.hidden_size).to(x.device)
        if self.cell_type == 'lstm':
            c0 = torch.zeros(self.rnn.num_layers, x.size(0), self.rnn.hidden_size).to(x.device)
            out, hidden = self.rnn(x, (h0, c0))
        else:
            out, hidden = self.rnn(x, h0)
        out = self.fc1(out[:, -1, :])
        out = nn.functional.relu(out)
        out = self.fc2(out)
        return out


## 3. Hàm phụ trợ — dựng dataset, loss chưng cất (KD), early stopping

Y hệt bản tái hiện (`utils/utils.py`, `exp/base_exp.py`), không đổi logic FGL.

In [ ]:
def create_time_series_dataset(data, lookback_window, forecasting_horizon, num_bins,
                               val_size, test_size, offset=0, MSE=False, batch_size=1):
    """Dung y het ham trong utils/utils.py cua ban tai hien."""
    x = np.array([pt[0] for pt in data])
    y_arr = np.array([pt[1] for pt in data])
    X_windows, y_windows = [], []
    for i in range(len(x) - lookback_window - forecasting_horizon + 1):
        X_windows.append(x[i: i + lookback_window])
        y_windows.append(y_arr[i + lookback_window + forecasting_horizon - 1])

    X = np.stack(X_windows)
    y_arr = np.stack(y_windows)

    N = X.shape[0]
    assert 0 < val_size + test_size < 1, "val_size + test_size must be in (0,1)"

    n_test = int(N * test_size)
    n_val = int(N * val_size)
    n_train = N - n_val - n_test

    X_train, X_val, X_test = X[:n_train], X[n_train:n_train + n_val], X[-n_test:]
    y_train, y_val, y_test = y_arr[:n_train], y_arr[n_train:n_train + n_val], y_arr[-n_test:]

    if not MSE:
        bin_edges = np.linspace(y_train.min(), y_train.max(), num_bins - 1)
        X_train = np.digitize(X_train, bin_edges)
        X_val = np.digitize(X_val, bin_edges)
        X_test = np.digitize(X_test, bin_edges)
        y_train = np.digitize(y_train, bin_edges)
        y_val = np.digitize(y_val, bin_edges)
        y_test = np.digitize(y_test, bin_edges)

    def to_tuples(X_arr, y_a):
        tup = [(i, X_arr[i], y_a[i]) for i in range(len(X_arr))]
        return tup[offset:] if offset else tup

    train_tuples = to_tuples(X_train, y_train)
    val_tuples = to_tuples(X_val, y_val)
    test_tuples = to_tuples(X_test, y_test)

    train_loader = DataLoader(train_tuples, batch_size=batch_size, shuffle=False, drop_last=True)
    val_loader = DataLoader(val_tuples, batch_size=batch_size, shuffle=False, drop_last=True)
    test_loader = DataLoader(test_tuples, batch_size=batch_size, shuffle=False, drop_last=True)

    return train_loader, val_loader, test_loader, None, None


def KL(student_logits, teacher_logits, temperature, alpha):
    """(1-alpha) * T^2 * KL( softmax(teacher/T) || log_softmax(student/T) )"""
    log_p_s = F.log_softmax(student_logits / temperature, dim=1)
    p_t = F.softmax(teacher_logits / temperature, dim=1)
    kd = F.kl_div(log_p_s, p_t, reduction='batchmean') * (temperature ** 2)
    return (1.0 - alpha) * kd


class EarlyStopper:
    def __init__(self, patience=5, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.counter = 0
        self.best_state = None

    def step(self, current_loss, model):
        if current_loss + self.min_delta < self.best_loss:
            self.best_loss = current_loss
            self.counter = 0
            self.best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            return False
        self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        if self.best_state:
            model.load_state_dict(self.best_state)


## 4. Vòng huấn luyện Teacher → Baseline → Student (FGL)

Y hệt `train_student_model` trong `exp/base_exp.py` — chỉ khác: đọc `data` từ biến trong bộ
nhớ (Mục 1) thay vì `data.pkl`, và nhận thêm tham số `arch` (phần cải tiến ở Mục 2).

In [ ]:
def train_student_model(student_horizon, alpha, num_bins, val_size, test_size, epochs,
                        temperature, lookback_window, batch_size,
                        patience=5, seed=42, arch='rnn', verbose=False):
    torch.manual_seed(seed)

    hidden_size = 128
    output_size = num_bins
    num_layers = 2
    lr = 1e-4

    teacher_train, teacher_val, teacher_test, _, _ = create_time_series_dataset(
        data=data, lookback_window=lookback_window, forecasting_horizon=1,
        num_bins=num_bins, val_size=val_size, test_size=test_size,
        offset=student_horizon - 1, batch_size=batch_size,
    )
    student_train, student_val, student_test, _, _ = create_time_series_dataset(
        data=data, lookback_window=lookback_window, forecasting_horizon=student_horizon,
        num_bins=num_bins, val_size=val_size, test_size=test_size,
        offset=0, batch_size=batch_size,
    )

    mse_fn = nn.MSELoss()
    celoss = nn.CrossEntropyLoss()

    # ---- Teacher (1-step) ----
    teacher = RNN(lookback_window, hidden_size, output_size, num_layers, cell_type=arch).to(device)
    opt_t = torch.optim.Adam(teacher.parameters(), lr=lr)
    stop_t = EarlyStopper(patience=patience)
    for epoch in range(epochs):
        teacher.train()
        for _, x, yb in teacher_train:
            x = x.float().to(device).view(-1, 1, lookback_window)
            yb = yb.long().to(device)
            loss = celoss(teacher(x), yb)
            opt_t.zero_grad(); loss.backward(); opt_t.step()
        teacher.eval()
        with torch.no_grad():
            val_loss = 0.
            for _, x, yb in teacher_val:
                x = x.float().to(device).view(-1, 1, lookback_window)
                yb = yb.long().to(device)
                val_loss += celoss(teacher(x), yb).item()
            val_loss /= len(teacher_val)
        if stop_t.step(val_loss, teacher):
            if verbose: print(f"[Teacher] Early stopping at epoch {epoch + 1}")
            break
    stop_t.restore(teacher)

    # ---- Baseline (H-step, KHÔNG FGL) ----
    baseline = RNN(lookback_window, hidden_size, output_size, num_layers, cell_type=arch).to(device)
    opt_b = torch.optim.Adam(baseline.parameters(), lr=lr)
    stop_b = EarlyStopper(patience=patience)
    for epoch in range(epochs):
        baseline.train()
        for _, x, yb in student_train:
            x = x.float().to(device).view(-1, 1, lookback_window)
            yb = yb.long().to(device)
            loss = celoss(baseline(x), yb)
            opt_b.zero_grad(); loss.backward(); opt_b.step()
        baseline.eval()
        with torch.no_grad():
            val_loss = 0.
            for _, x, yb in student_val:
                x = x.float().to(device).view(-1, 1, lookback_window)
                yb = yb.long().to(device)
                val_loss += celoss(baseline(x), yb).item()
            val_loss /= len(student_val)
        if stop_b.step(val_loss, baseline):
            if verbose: print(f"[Baseline] Early stopping at epoch {epoch + 1}")
            break
    stop_b.restore(baseline)

    # ---- Student + chưng cất tri thức (CÓ FGL) ----
    student = RNN(lookback_window, hidden_size, output_size, num_layers, cell_type=arch).to(device)
    opt_s = torch.optim.Adam(student.parameters(), lr=lr)
    stop_s = EarlyStopper(patience=patience)
    for epoch in range(epochs):
        student.train()
        for (_, x_s, y_s), (_, x_t, _y_t) in zip(student_train, teacher_train):
            x_s = x_s.float().to(device).view(-1, 1, lookback_window)
            targets = y_s.long().to(device)
            outputs = student(x_s)
            x_t = x_t.float().to(device).view(-1, 1, lookback_window)
            with torch.no_grad():
                logits = teacher(x_t)
            loss = alpha * celoss(outputs, targets) + KL(outputs, logits, temperature, alpha)
            opt_s.zero_grad(); loss.backward(); opt_s.step()
        student.eval()
        with torch.no_grad():
            val_loss = 0.
            for _, x, yb in student_val:
                x = x.float().to(device).view(-1, 1, lookback_window)
                yb = yb.long().to(device)
                val_loss += celoss(student(x), yb).item()
            val_loss /= len(student_val)
        if stop_s.step(val_loss, student):
            if verbose: print(f"[Student] Early stopping at epoch {epoch + 1}")
            break
    stop_s.restore(student)

    # ---- Đánh giá (Test MSE) ----
    def evaluate(model, loader):
        model.eval()
        tot = 0.
        with torch.no_grad():
            for _, x, yb in loader:
                x = x.float().to(device).view(-1, 1, lookback_window)
                tot += mse_fn(model(x).argmax(dim=1).float(), yb.float().to(device).squeeze(-1)).item()
        return tot / len(loader)

    teacher_mse = evaluate(teacher, teacher_test)
    baseline_mse = evaluate(baseline, student_test)
    student_mse = evaluate(student, student_test)

    return teacher_mse, baseline_mse, student_mse


## 5. Chạy so sánh 3 kiến trúc × 6 seed

Dùng đúng tham số mẫu tác giả gợi ý (`horizon=5, num_bins=50, alpha=0.5, epochs=20`) và
đúng 6 seed đã log trong `EXPERIMENT_LOG.md` (`0,1,2,3,4,42`) để đối chiếu được trực tiếp.

**Ước lượng thời gian**: trên máy local (CPU, không GPU), mỗi lần chạy mất ~2-5 phút
(`batch_size=1` khiến mỗi epoch lặp qua từng mẫu một, khá chậm) → tổng 18 lần chạy có thể
mất **60-100 phút**. Trên Kaggle có thể nhanh/chậm hơn tuỳ phần cứng được cấp — **muốn ước
lượng trước**, rút gọn `SEEDS` xuống còn `[0, 42]` để chạy thử nhanh (6 lần thay vì 18), rồi
mở rộng lại `[0,1,2,3,4,42]` khi đã biết thời gian/lần chạy.

In [ ]:
ARCHS = ["rnn", "lstm", "gru"]
SEEDS = [0, 1, 2, 3, 4, 42]   # rut gon vd [0, 42] de chay thu nhanh truoc

HORIZON, NUM_BINS, ALPHA, EPOCHS, TEMP = 5, 50, 0.5, 20, 4
LOOKBACK, BATCH = 1, 1

records = []
t0 = time.time()
for arch in ARCHS:
    for seed in SEEDS:
        t1 = time.time()
        teacher_mse, baseline_mse, student_mse = train_student_model(
            student_horizon=HORIZON, alpha=ALPHA, num_bins=NUM_BINS,
            val_size=0.2, test_size=0.2, epochs=EPOCHS,
            temperature=TEMP, lookback_window=LOOKBACK, batch_size=BATCH,
            seed=seed, arch=arch,
        )
        reduction_pct = (baseline_mse - student_mse) / baseline_mse * 100
        records.append(dict(arch=arch, seed=seed, teacher_mse=teacher_mse,
                             baseline_mse=baseline_mse, student_mse=student_mse,
                             reduction_pct=reduction_pct))
        print(f"[{arch:4s} seed={seed:>2}] Teacher={teacher_mse:7.4f} "
              f"Baseline={baseline_mse:8.4f} Student={student_mse:8.4f} "
              f"Giảm={reduction_pct:5.2f}%   ({time.time() - t1:.0f}s)")

print(f"\nTổng thời gian: {(time.time() - t0) / 60:.1f} phút")
df = pd.DataFrame(records)
df


## 6. Tổng hợp & biểu đồ so sánh

In [ ]:
summary = df.groupby("arch")["reduction_pct"].agg(["mean", "std", "min", "max"]).reindex(ARCHS)
summary.columns = ["Giảm MSE TB (%)", "Std (%)", "Min (%)", "Max (%)"]
summary


In [ ]:
# Mau categorical da validate CVD-safe (all-pairs, skill dataviz) - thu tu co dinh RNN/LSTM/GRU
ARCH_LABELS = {"rnn": "RNN (gốc)", "lstm": "LSTM", "gru": "GRU"}
ARCH_COLORS = {"rnn": "#2a78d6", "lstm": "#eb6834", "gru": "#1baf7a"}

means = summary["Giảm MSE TB (%)"].values
stds = summary["Std (%)"].values
labels = [ARCH_LABELS[a] for a in ARCHS]
colors = [ARCH_COLORS[a] for a in ARCHS]

fig, ax = plt.subplots(figsize=(6, 4.5), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")

bars = ax.bar(labels, means, yerr=stds, capsize=4, width=0.55,
              color=colors, edgecolor="#fcfcfb", linewidth=2,
              error_kw=dict(ecolor="#52514e", elinewidth=1.2))

for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.0,
            f"{m:.1f}%", ha="center", va="bottom",
            fontsize=11, color="#0b0b0b", fontweight="bold")

ax.set_ylabel("Giảm MSE trung bình, 6 seed (%)", color="#52514e", fontsize=10)
ax.set_title("So sánh kiến trúc: RNN vs LSTM vs GRU\n(Mackey-Glass, Future-Guided Learning)",
             color="#0b0b0b", fontsize=12, fontweight="bold", pad=12)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color("#c3c2b7")
ax.tick_params(colors="#898781")
ax.yaxis.grid(True, color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
ax.set_ylim(0, max(means) + max(stds) + 6)

plt.tight_layout()
plt.savefig("arch_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


## Ghi chú

- **Seed sinh dữ liệu** (Mục 1, `seed_id=42`) và **seed huấn luyện** (Mục 5, `SEEDS`) là hai
  khái niệm khác nhau — đúng thiết kế gốc của tác giả: 1 chuỗi Mackey-Glass duy nhất, train
  lại nhiều lần với seed khởi tạo trọng số khác nhau.
- File `arch_comparison.png` được lưu lại trong working directory của notebook (Kaggle:
  thư mục `/kaggle/working/`) — tải về để chèn thẳng vào slide/báo cáo.
- Đây là **kết quả tự đề xuất cải tiến**, không nằm trong paper gốc hay code tác giả —
  nên trình bày trong báo cáo kèm ghi chú rõ nguồn gốc (xem
  `repro/ours/patches/README.md` và `repro/docs/EXPERIMENT_LOG.md`, mục "Lần chạy 4").
